<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB15_Transfer_Learning_Feature_Extraction_vs_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB15 · Class 15 — Transfer Learning: Feature Extraction vs. Fine-Tuning**

## Block 3: AI — Deep Learning (continued)

`NB13` trained a CNN completely from scratch on ~400 real underwater images — a small dataset by deep-learning standards, and `its filters had to learn everything, including basic edges and textures, from that small sample alone`. This class asks: what if we didn't start from scratch? **Transfer learning** reuses a network already trained on millions of unrelated photos, on the reasonable assumption (backed by `NB13`'s closing reference to Zeiler & Fergus) that its early, generic filters are useful for almost any image task. We compare the two standard strategies — **feature extraction** and **fine-tuning** — directly against each other, and against `NB13`'s from-scratch result, on the exact same real LIACi marine-growth classification task.

### Learning objectives

By the end of this class, students will be able to:
- Explain what transfer learning is and why it works, in terms of what different CNN layers learn.
- Distinguish feature extraction (frozen backbone) from fine-tuning (partially unfrozen backbone), and explain the tradeoff between them.
- Load and correctly preprocess data for a real pretrained model (`torchvision`'s `ResNet18`).
- Freeze and selectively unfreeze layers of a pretrained PyTorch model.
- Compare from-scratch training, feature extraction, and fine-tuning fairly, on the same real data and evaluation.

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap, today's roadmap | 5 min | Theory |
| 2 | What is transfer learning, and why does it work? | 10 min | Theory |
| 3 | Two strategies: feature extraction vs. fine-tuning | 15 min | Theory + Practice |
| 4 | Loading a real pretrained model | 15 min | Practice |
| 5 | Real dataset: reusing NB13's (fixed) LIACi pipeline | 15 min | Practice |
| 6 | Preprocessing for a pretrained model | 10 min | Theory + Practice |
| 7 | Hands-on: feature extraction | 15 min | Practice |
| 8 | Hands-on: fine-tuning | 15 min | Practice |
| 9 | Comparing all three approaches | 15 min | Practice |
| 10 | Summary, homework, next class | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.

> **Before we start**: like `NB13`, this class downloads the real ~1 GB LIACi dataset and trains on real images. Switch to a GPU runtime now if available (`Runtime → Change runtime type → T4 GPU`).

---

## 1. Recap: where we are

- **`NB13`**: a CNN trained from scratch on real underwater images — every filter learned from ~400 examples alone.
- **`NB14`**: RNN/LSTM theory and a real sequence-forecasting model.
- **`NB15`** (today): reusing a network already trained on millions of images, instead of starting from nothing.

---

## 2. What is transfer learning, and why does it work?

**[Transfer learning](https://en.wikipedia.org/wiki/Transfer_learning)** takes a model trained on one task (usually a huge, general dataset) and reuses some or all of its learned weights as the starting point for a different, usually smaller, task — instead of initializing every weight randomly, as `NB13`'s `HullCNN` did.

Why this works for CNNs specifically, tying directly back to `NB13` §9's feature-map visualization: a network trained on **ImageNet** (1.4 million natural photos across 1,000 categories — cats, cars, furniture, everything but underwater hull imagery) still learns, in its earliest layers, `filters that detect edges, color gradients, and simple textures`. Those are useful for recognizing *almost any* visual pattern, underwater hull conditions included — only the later, more task-specific layers need to change much. Reusing the early layers means our small ~400-image dataset only has to teach the network what's *different* about our task, not how to see at all.

---

## 3. Two strategies: feature extraction vs. fine-tuning

| | Feature extraction | [Fine-tuning](https://en.wikipedia.org/wiki/Fine-tuning_%28deep_learning%29) |
|---|---|---|
| Backbone weights | **Frozen** — never updated | **Partially unfrozen** — some layers continue training |
| What gets trained | Only a new final layer (the classifier head) | The new head **and** some of the backbone's later layers |
| Training speed | Fast — most of the network does no backward pass | Slower — more parameters being updated |
| Data needed | Works with very little data | Needs somewhat more data, or careful learning rates, to avoid overfitting or "forgetting" |
| When to prefer it | Your task is similar to the pretraining data, or your dataset is tiny | Your task differs more from the pretraining data, and you have enough data to safely adapt more of the network |

Let's visualize which layers are touched by each strategy:

Draw both strategies side by side, frozen vs. trainable blocks:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

block_labels = ["conv\nblock 1", "conv\nblock 2", "conv\nblock 3", "conv\nblock 4", "avgpool", "fc\n(new)"]
feature_extraction_frozen = [True, True, True, True, True, False]
fine_tuning_frozen = [True, True, False, False, False, False]

for ax, title, frozen_pattern in zip(
    axes, ["Feature extraction", "Fine-tuning"], [feature_extraction_frozen, fine_tuning_frozen]
):
    for i, (label, frozen) in enumerate(zip(block_labels, frozen_pattern)):
        color = "lightgray" if frozen else "lightcoral"
        ax.add_patch(patches.Rectangle((i, 0), 0.9, 1, facecolor=color, edgecolor="black"))
        ax.text(i + 0.45, 0.5, label, ha="center", va="center", fontsize=8)
        ax.text(i + 0.45, -0.3, "frozen" if frozen else "trainable", ha="center", fontsize=7,
                color="gray" if frozen else "darkred")
    ax.set_xlim(-0.3, len(block_labels))
    ax.set_ylim(-0.6, 1.3)
    ax.axis("off")
    ax.set_title(title)

plt.tight_layout()
plt.show()

Fine-tuning always unfreezes the *later* layers (closer to the output), never the earliest ones — `those earliest, most generic edge/texture filters are exactly the part transfer learning trusts the most`.

---

## 4. Loading a real pretrained model

`torchvision` ships several models pretrained on ImageNet. We'll use **ResNet18** — a real, widely-used architecture, small enough to fine-tune quickly on a CPU if needed:

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights)

print(resnet.fc)  # the final layer -- built for ImageNet's 1,000 classes
sum(p.numel() for p in resnet.parameters())

That parameter count dwarfs `NB13`'s `HullCNN` — but we are about to freeze almost all of it, so the *effective* number of weights we train ourselves will end up far smaller, not larger.

**Try it yourself**: `layer4` in Sections 7-8 refers to one of several named blocks. List them to see the full structure available to freeze or unfreeze:

In [ ]:
for name, _ in resnet.named_children():
    print(name)


---

## 5. Real dataset: reusing `NB13`'s (fixed) LIACi pipeline

Same real dataset, same target condition (marine growth), same label-building logic as `NB13` — including the fix for LIACi's `.bmp` mask files (which don't share the source `.jpg` images' extension):

In [ ]:
!wget -q -O liaci_data.zip https://liaci.sintef.cloud/download_data/data.zip
!unzip -oq liaci_data.zip

import os
from PIL import Image
import numpy as np

base_dir = "LIACi_dataset_pretty"
imgs_path = os.path.join(base_dir, "images")
masks_path = os.path.join(base_dir, "masks")

image_files = sorted(os.listdir(imgs_path))
target_class = "marine_growth"

def label_for(fname, cls):
    stem = os.path.splitext(fname)[0]
    mask_file = os.path.join(masks_path, cls, stem + ".bmp")
    if not os.path.exists(mask_file):
        return 0
    mask = np.array(Image.open(mask_file).convert("L"))
    return int(mask.max() > 0)

all_labels = {f: label_for(f, target_class) for f in image_files}
print(f"{sum(all_labels.values())} / {len(all_labels)} images show '{target_class}'")

Balanced subsample, exactly as in `NB13`:

In [ ]:
import random

random.seed(42)
positive_files = [f for f in image_files if all_labels[f] == 1]
negative_files = [f for f in image_files if all_labels[f] == 0]

n_per_class = min(len(positive_files), len(negative_files), 200)
sample_files = random.sample(positive_files, n_per_class) + random.sample(negative_files, n_per_class)
random.shuffle(sample_files)
print("Sample size:", len(sample_files))

---

## 6. Preprocessing for a pretrained model

`NB13` resized images to 96×96 and normalized pixels to 0–1 — a choice we made freely, since `HullCNN` was ours to design. A **pretrained** model is different: `it was trained expecting images preprocessed a very specific way` (typically 224×224, normalized with ImageNet's exact per-channel mean and standard deviation), and feeding it anything else quietly degrades performance, since every pixel value would sit in a different range than what the frozen filters learned to expect.

`torchvision`'s modern API stores the exact preprocessing a given set of weights expects, so we don't have to hardcode it ourselves:

In [ ]:
preprocess = weights.transforms()
print(preprocess)

Use it directly to build our image tensors — same real files, correctly preprocessed this time for a pretrained network:

In [ ]:
def load_and_preprocess(fname):
    img = Image.open(os.path.join(imgs_path, fname)).convert("RGB")
    return preprocess(img)

X_img = torch.stack([load_and_preprocess(f) for f in sample_files])
y_img = torch.tensor([all_labels[f] for f in sample_files], dtype=torch.float32).view(-1, 1)
X_img.shape

Split 60/20/20 by index, stratified, exactly like every classification task since `NB08`:

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

labels_arr = y_img.numpy().ravel()
idx = np.arange(len(labels_arr))
idx_train_full, idx_test = train_test_split(idx, test_size=0.2, random_state=42, stratify=labels_arr)
idx_train, idx_val = train_test_split(
    idx_train_full, test_size=0.25, random_state=42, stratify=labels_arr[idx_train_full]
)

X_train, y_train = X_img[idx_train], y_img[idx_train]
X_val, y_val = X_img[idx_val], y_img[idx_val]
X_test, y_test = X_img[idx_test], y_img[idx_test]
X_train.shape, X_val.shape, X_test.shape

---

## 7. Hands-on: feature extraction

Freeze every backbone parameter, then replace the final layer with a fresh one sized for our binary task — the new layer is trainable by default, since it's never had `requires_grad` set to `False`:

In [ ]:
fe_model = models.resnet18(weights=weights)

for param in fe_model.parameters():
    param.requires_grad = False

fe_model.fc = nn.Linear(fe_model.fc.in_features, 1)

trainable = sum(p.numel() for p in fe_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in fe_model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

Only the new head is trainable — compare that count to `NB13`'s `HullCNN` parameter count from that class's Part 8. Train with the same recipe as every classifier since `NB11`:

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, fe_model.parameters()), lr=0.001)

n_epochs = 10
fe_train_losses, fe_val_losses = [], []

for epoch in range(n_epochs):
    fe_model.train()
    optimizer.zero_grad()
    loss = criterion(fe_model(X_train), y_train)
    loss.backward()
    optimizer.step()
    fe_train_losses.append(loss.item())

    fe_model.eval()
    with torch.no_grad():
        fe_val_losses.append(criterion(fe_model(X_val), y_val).item())

    print(f"Epoch {epoch + 1}/{n_epochs} - train: {fe_train_losses[-1]:.4f} - val: {fe_val_losses[-1]:.4f}")

Note we needed far fewer epochs than `NB13`'s from-scratch training — the backbone already "knows how to see"; `the new head only has to learn a simple boundary on top of features that are already meaningful`.

---

## 8. Hands-on: fine-tuning

Same starting point, but this time unfreeze `layer4` (ResNet18's last, most task-specific residual block) in addition to the new head — matching the diagram from Part 3:

In [ ]:
ft_model = models.resnet18(weights=weights)

for param in ft_model.parameters():
    param.requires_grad = False
for param in ft_model.layer4.parameters():
    param.requires_grad = True

ft_model.fc = nn.Linear(ft_model.fc.in_features, 1)

trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in ft_model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

More trainable parameters than Part 7, but still a small fraction of the full network. One important detail: fine-tuning conventionally uses a **smaller learning rate** than training from scratch — we're nudging already-good pretrained weights, not learning from nothing, and large updates risk destroying what they already learned:

In [ ]:
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, ft_model.parameters()), lr=0.0001)

ft_train_losses, ft_val_losses = [], []

for epoch in range(n_epochs):
    ft_model.train()
    optimizer.zero_grad()
    loss = criterion(ft_model(X_train), y_train)
    loss.backward()
    optimizer.step()
    ft_train_losses.append(loss.item())

    ft_model.eval()
    with torch.no_grad():
        ft_val_losses.append(criterion(ft_model(X_val), y_val).item())

    print(f"Epoch {epoch + 1}/{n_epochs} - train: {ft_train_losses[-1]:.4f} - val: {ft_val_losses[-1]:.4f}")

Plot both training runs together:

In [ ]:
plt.plot(fe_val_losses, label="Feature extraction (val)")
plt.plot(ft_val_losses, label="Fine-tuning (val)")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Feature extraction vs. fine-tuning")
plt.legend()
plt.show()

---

## 9. Comparing all three approaches

Evaluate both on the untouched test set:

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

results = []
for name, model in [("Feature extraction", fe_model), ("Fine-tuning", ft_model)]:
    model.eval()
    with torch.no_grad():
        preds = (torch.sigmoid(model(X_test)) > 0.5).float()
    y_test_np = y_test.numpy().ravel()
    preds_np = preds.numpy().ravel()
    results.append({
        "Strategy": name,
        "Test accuracy": accuracy_score(y_test_np, preds_np),
        "Test F1": f1_score(y_test_np, preds_np, zero_division=0),
    })

import pandas as pd
pd.DataFrame(results)

A bar chart makes the accuracy comparison easier to read at a glance:

In [ ]:
results_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(results_df["Strategy"], results_df["Test accuracy"], color=["steelblue", "darkorange"])
ax.set_ylabel("Test accuracy")
ax.set_title("Feature extraction vs. fine-tuning")
ax.set_ylim(0, 1)
plt.show()


Accuracy and F1 summarize; a confusion matrix for each strategy shows *which* errors each one makes:

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, model) in zip(axes, [("Feature extraction", fe_model), ("Fine-tuning", ft_model)]):
    model.eval()
    with torch.no_grad():
        model_preds = (torch.sigmoid(model(X_test)) > 0.5).float().numpy().ravel()
    cm = confusion_matrix(y_test.numpy().ravel(), model_preds, labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["Absent", "Present"]).plot(cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()


**Try it yourself**: look at real predictions side by side with the actual test images (same qualitative check `NB13` used) — do the fine-tuned model's mistakes look reasonable to a human eye?

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axes.ravel(), range(8)):
    img_show = X_test[i].permute(1, 2, 0).numpy()
    img_show = (img_show - img_show.min()) / (img_show.max() - img_show.min())  # undo ImageNet normalization for display
    ax.imshow(img_show)
    ax.set_title(f"True: {int(y_test_np[i])}  Pred: {int(preds_np[i])}")
    ax.axis("off")
plt.suptitle("Fine-tuned model -- real predictions on real test images")
plt.tight_layout()
plt.show()


**Build the full comparison yourself**: add a third row to the table above using `NB13`'s test accuracy for the from-scratch `HullCNN` (re-run that notebook if you don't have the number handy). With only ~240 training images, which approach actually won here — and does that match the general expectation that transfer learning helps most when your own dataset is small relative to what the pretrained model saw? If feature extraction and fine-tuning finished close together, that's a reasonable outcome too: `layer4`'s pretrained ImageNet features may already have been "good enough" for this task, leaving little room for fine-tuning's extra flexibility to help on this little data — a legitimate, informative result, not a failure of the notebook (the same honest framing `NB14` used for its own baseline comparison).

---

## Class summary

- Transfer learning reuses a network already trained on a large, general dataset (ResNet18 on ImageNet) as a starting point, instead of learning everything from scratch.
- It works because early CNN layers learn generic features (edges, textures) useful across nearly any image task — `NB13`'s feature-map visualization made this concrete before we relied on it here.
- Feature extraction freezes the whole backbone and trains only a new head — fast, few trainable parameters, good for very small datasets.
- Fine-tuning also unfreezes some later backbone layers, usually with a smaller learning rate — more flexible, needs more data to pay off safely.
- Pretrained models expect a specific preprocessing pipeline (`weights.transforms()`) — feeding them `NB13`-style raw 0–1 pixels would silently hurt performance.
- We compared feature extraction, fine-tuning, and (by reference) `NB13`'s from-scratch training on the same real task, rather than trusting any one number in isolation.

## For the next class (NB16)

**Autoencoders and anomaly detection** with Deep Learning — an unsupervised counterpart to this block's classifiers, learning to reconstruct "normal" data well enough that anything reconstructed poorly stands out as anomalous.

## Homework / Practice Ideas

1. Unfreeze `layer3` as well as `layer4` in Part 8 and retrain — does test accuracy improve, or does the extra flexibility start to overfit on this small dataset?
2. Try a different pretrained backbone, `models.resnet34(weights=ResNet34_Weights.DEFAULT)` (same pattern, different weights class) — does a bigger pretrained network help or just slow training down here?
3. In Part 7, raise the learning rate from `0.001` to `0.01` — does feature extraction still train stably? Now try the same change in Part 8's fine-tuning optimizer — does *that* still train stably? What does the difference tell you about why fine-tuning conventionally uses a smaller learning rate?
4. Using the printed `preprocess` object from Part 6, find the actual resize dimensions and normalization values it applies — do they match the "typically 224×224, ImageNet mean/std" description from that section?
5. Re-run Part 9's comparison using `weather_conditions` or a different target class from `NB13`'s mask list instead of marine growth — does the relative ranking of the three approaches change?

> ***As always: when two model variants score close to each other, the more useful homework is explaining *why* they're close, not just picking whichever number is marginally higher.***
